# 10 — Character Clustering and Cross-Page Matching

Convert visual similarities into stable sequence-local speaker IDs.

> **Course principle:** every notebook separates **general-purpose knowledge** from **Manga Task application**.

## Clustering baseline

Agglomerative clustering is convenient when the number of characters is unknown. DBSCAN can mark outliers but is sensitive to distance thresholds.

In [ ]:
import numpy as np
from sklearn.cluster import AgglomerativeClustering

X = np.random.randn(12, 32)

clusterer = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=4.0
)
labels = clusterer.fit_predict(X)
print(labels)

## Incremental identity registry

A practical sequence-level system can maintain a small registry:

`char1 → prototype embedding`
`char2 → prototype embedding`

Each new character crop is matched to the closest prototype; if similarity is too low, create a new identity.

In [ ]:
import numpy as np

def cosine(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(a @ b)

def assign_identity(vec, registry, threshold=0.70):
    if not registry:
        registry["char1"] = vec
        return "char1"

    scores = {k: cosine(vec, v) for k, v in registry.items()}
    best = max(scores, key=scores.get)

    if scores[best] >= threshold:
        registry[best] = 0.8 * registry[best] + 0.2 * vec
        return best

    new_id = f"char{len(registry)+1}"
    registry[new_id] = vec
    return new_id

## Speaker assignment is not identity matching

Identity matching tells you who a crop belongs to. Speaker assignment asks which visible or off-panel character produced a specific balloon.

You may need bubble-tail geometry, proximity, panel context, conversational alternation, or a VLM decision on top of identity tracking.

## Project experiment

Measure identity consistency separately from transcription. A perfect OCR system can still have a poor joint score if it changes labels across pages.